In [1]:
from deepfake_detection.config import DATA_RAW, MODELS_DIR
print(DATA_RAW, DATA_RAW.exists())
print(MODELS_DIR, MODELS_DIR.exists())

C:\Users\21624\Desktop\deepfake_detection\data\raw True
C:\Users\21624\Desktop\deepfake_detection\models True


In [3]:
%%writefile ../deepfake_detection/deepfake_detection/utils.py
import torch
import timm
from torchvision import transforms as T
from PIL import Image
from .config import MODELS_DIR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARCH = "efficientnet_b0"
NUM_CLASSES = 2

TRANSFORM = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def load_model(weights_name):
    model = timm.create_model(ARCH, pretrained=False, num_classes=NUM_CLASSES)
    state = torch.load(MODELS_DIR / weights_name, map_location=DEVICE)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

def predict(model, image_path):
    img = Image.open(image_path).convert("RGB")
    x = TRANSFORM(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    return {"fake": probs[0].item(), "real": probs[1].item()}

Overwriting ../deepfake_detection/deepfake_detection/utils.py


In [2]:
from deepfake_detection.utils import load_model

m_cifake = load_model("model_cifake.pth")
m_faces  = load_model("model_faces.pth")
print("both models loaded")

C:\Users\21624\anaconda3\envs\deepfake\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


both models loaded


In [5]:
from deepfake_detection.config import DATA_RAW

for p in sorted(DATA_RAW.rglob("*")):
    if p.is_dir():
        names = {c.name.lower() for c in p.iterdir() if c.is_dir()}
        if {"fake", "real"} <= names:
            print(p)

C:\Users\21624\Desktop\deepfake_detection\data\raw\Facesai vs real\real_vs_fake\real-vs-fake\test
C:\Users\21624\Desktop\deepfake_detection\data\raw\Facesai vs real\real_vs_fake\real-vs-fake\train
C:\Users\21624\Desktop\deepfake_detection\data\raw\Facesai vs real\real_vs_fake\real-vs-fake\valid
C:\Users\21624\Desktop\deepfake_detection\data\raw\images ai vs real\test
C:\Users\21624\Desktop\deepfake_detection\data\raw\images ai vs real\train


In [1]:
import numpy as np, pandas as pd, torch, random
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, recall_score

from deepfake_detection.config import DATA_RAW, DATA_PROCESSED, ROOT, SEED
from deepfake_detection.utils import load_model, TRANSFORM, DEVICE

DATASETS = {
    "cifake": DATA_RAW / "images ai vs real" / "test",
    "faces":  DATA_RAW / "Facesai vs real" / "real_vs_fake" / "real-vs-fake" / "test",
}
MODELS = {
    "model_cifake":   "model_cifake.pth",
    "model_faces":    "model_faces.pth",
    "model_combined": "model_combined.pth",
}
MAX_PER_CLASS = 1000   # lower to 300 if too slow on CPU

class FakeRealDataset(Dataset):
    """label 0 = fake, 1 = real (same order as in Kaggle)"""
    def __init__(self, root, max_per_class=None):
        self.items = []
        subs = {p.name.lower(): p for p in Path(root).iterdir() if p.is_dir()}
        rng = random.Random(SEED)
        for label, name in [(0, "fake"), (1, "real")]:
            files = sorted(f for f in subs[name].rglob("*")
                           if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
            if max_per_class and len(files) > max_per_class:
                files = rng.sample(files, max_per_class)
            self.items += [(f, label) for f in files]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return TRANSFORM(Image.open(path).convert("RGB")), label

datasets_loaded = {}
for name, root in DATASETS.items():
    datasets_loaded[name] = FakeRealDataset(root, MAX_PER_CLASS)
    print(name, len(datasets_loaded[name]), "images")

C:\Users\21624\anaconda3\envs\deepfake\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cifake 2000 images
faces 2000 images


In [2]:
def evaluate(model, dataset):
    dl = DataLoader(dataset, batch_size=64, num_workers=0)
    p_fake, y_true = [], []
    with torch.no_grad():
        for x, y in dl:
            probs = torch.softmax(model(x.to(DEVICE)), dim=1)
            p_fake += probs[:, 0].cpu().tolist()     # index 0 = fake
            y_true += y.tolist()
    y_true, p_fake = np.array(y_true), np.array(p_fake)
    is_fake = (y_true == 0).astype(int)              # positive class = fake
    pred_fake = (p_fake >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(is_fake, pred_fake).ravel()
    return {
        "AUC": roc_auc_score(is_fake, p_fake),
        "Accuracy": accuracy_score(is_fake, pred_fake),
        "Recall (fake)": recall_score(is_fake, pred_fake),
        "False positive rate": fp / (fp + tn),       # real images flagged as fake
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    }

rows = []
for m_name, weights in MODELS.items():
    model = load_model(weights)
    for d_name, ds in datasets_loaded.items():
        r = evaluate(model, ds)
        r.update({"model": m_name, "tested_on": d_name})
        rows.append(r)
        print(f"{m_name} -> {d_name}: AUC={r['AUC']:.3f}")

results = pd.DataFrame(rows).set_index(["model", "tested_on"])
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
results.to_csv(DATA_PROCESSED / "evaluation_results.csv")
results.round(3)

model_cifake -> cifake: AUC=0.983
model_cifake -> faces: AUC=0.460
model_faces -> cifake: AUC=0.493
model_faces -> faces: AUC=0.880
model_combined -> cifake: AUC=0.992
model_combined -> faces: AUC=0.972


AUC  Accuracy  Recall (fake)  False positive rate  \
model          tested_on                                                        
model_cifake   cifake     0.983     0.936          0.937                0.065   
               faces      0.460     0.486          0.902                0.929   
model_faces    cifake     0.493     0.500          0.071                0.070   
               faces      0.880     0.792          0.785                0.200   
model_combined cifake     0.992     0.956          0.962                0.049   
               faces      0.972     0.905          0.904                0.094   

                           TP   FP   TN   FN  
model          tested_on                      
model_cifake   cifake     937   65  935   63  
               faces      902  929   71   98  
model_faces    cifake      71   70  930  929  
               faces      785  200  800  215  
model_combined cifake     962   49  951   38  
               faces      904   94  906   96

In [ ]:
from deepfake_detection.utils import load_model, predict

IMAGE_PATH = r"C:\Users\...\my_image.jpg"   # <- put your image path here

MODELS = ["model_cifake.pth", "model_faces.pth", "model_combined.pth"]

for weights in MODELS:
    model = load_model(weights)
    result = predict(model, IMAGE_PATH)
    print(f"{weights:20s} -> {result['fake']*100:.1f}% AI-generated | {result['real']*100:.1f}% real")